# Open Notebook & Additional Resources

<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ORM_AI_Agents_Bootcamp/blob/main/hands_on/DAY_2_HANDS_ON_SESSION_4_langgraph_logging_checkpointing.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://learning.oreilly.com/library/view/ai-agents-the/0642572247775/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="AI Agents Book – Read on O'Reilly"/>
</a>





#<font color="red" size="10">
<b>HANDS-ON TIME: 15 mins</b>
</font>

# About the Notebook

Hands-on: Log state transitions + inspect checkpoints


1. Run **one small logging function** that records each **state transition** (here: each node update from `stream_mode="updates"`).
2. **Run the workflow twice** with a **tiny change** in the user message.
3. **Compare** the logged steps and checkpoint history to reason about **stability** (same graph path or not, same tool usage or not).

This notebook uses:

- A minimal **LangGraph** loop: `llm` → optional `tools` → `llm` → …
- **`logging`** from the Python standard library (good for notebooks and services)
- **`InMemorySaver`** checkpointing ([persistence overview](https://docs.langchain.com/oss/python/langgraph/persistence)) so each `thread_id` keeps a replayable history you can inspect with `get_state_history`

> In production you typically swap `InMemorySaver` for **SQLite** or **Postgres** savers from the LangGraph checkpoint packages; the **compile + `thread_id` + history APIs stay the same**.

In [1]:
%pip install -q "langgraph>=0.2" "langchain-openai>=0.3" "langchain-core>=0.3" "python-dotenv>=1.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 33.2 MB/s eta 0:00:00


In [3]:
from __future__ import annotations

import logging
import os
from typing import Annotated, Any, List

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from typing_extensions import TypedDict

# Dedicated logger so we can tune verbosity without spam from other libraries.
LOG = logging.getLogger("workflow.debug")

In [4]:
def setup_workflow_logging(level: int = logging.INFO) -> None:
    """Send workflow transition lines to stderr with a readable format."""
    if not LOG.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
        LOG.addHandler(handler)
    LOG.setLevel(level)
    logging.getLogger("httpx").setLevel(logging.WARNING)


setup_workflow_logging()

In [5]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

NEBIUS_API_KEY = os.environ.get("NEBIUS_API_KEY")

if not NEBIUS_API_KEY:
    print("⚠️ NEBIUS_API_KEY not found.")
    NEBIUS_API_KEY = input("Enter your NEBIUS_API_KEY: ").strip()

NEBIUS_API_KEY = NEBIUS_API_KEY.strip().strip('"').strip("'")



## Simple tools

Two tiny tools are enough to force a **tool path** vs a **direct answer path**, which is what you usually care about when eyeballing stability.

In [6]:
@tool
def word_stats(text: str) -> dict:
    """Return character and word counts for a single line of text."""
    t = text.strip()
    return {"chars": len(t), "words": len(t.split()) if t else 0}


@tool
def add_small(a: int, b: int) -> int:
    """Add two small integers."""
    return int(a) + int(b)

In [7]:
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]


SYSTEM = (
    "You are a concise assistant. Tools: word_stats, add_small. "
    "Use word_stats when the user asks for counts on a specific string. "
    "Use add_small for explicit small-integer arithmetic. "
    "If no tool is needed, answer directly in one short paragraph."
)



llm = ChatOpenAI(
    model="openai/gpt-oss-120b",
    api_key=NEBIUS_API_KEY,
    base_url="https://api.tokenfactory.nebius.com/v1/",
    temperature=0,
).bind_tools(
    [word_stats, add_small],
    tool_choice="auto",
)


def llm_node(state: AgentState) -> AgentState:
    msgs = [SystemMessage(content=SYSTEM), *state["messages"]]
    return {"messages": [llm.invoke(msgs)]}


def route_after_llm(state: AgentState):
    last = state["messages"][-1]
    calls = getattr(last, "tool_calls", None) or []
    return "tools" if calls else END


tool_node = ToolNode(tools=[word_stats, add_small])

checkpointer = InMemorySaver()
workflow = StateGraph(AgentState)
workflow.add_node("llm", llm_node)
workflow.add_node("tools", tool_node)
workflow.add_edge(START, "llm")
workflow.add_conditional_edges("llm", route_after_llm, {"tools": "tools", END: END})
workflow.add_edge("tools", "llm")

app = workflow.compile(checkpointer=checkpointer)

## One logging helper + tracked run

- **`stream_mode="updates"`** yields one dict per graph step: `{node_name: partial_state_update}`.
- That is a practical definition of a **visible transition** for debugging (see also [DAY_1_SESSION_1_LLM_state](../demos/DAY_1_SESSION_1_LLM_state.ipynb) in this repo).

The helper below keeps the **exercise constraint**: a single function you can reuse anywhere you already have stream chunks.

In [8]:
def summarize_update(node: str, payload: Any) -> str:
    """Short, human-readable summary of one node's write to state."""
    if not isinstance(payload, dict):
        return repr(payload)[:240]

    if node == "llm":
        msgs = payload.get("messages") or []
        msg = msgs[-1] if msgs else None
        if isinstance(msg, AIMessage):
            calls = getattr(msg, "tool_calls", None) or []
            if calls:
                names = [
                    c.get("name", "?") if isinstance(c, dict) else getattr(c, "name", "?")
                    for c in calls
                ]
                return f"llm → tool_calls={names}"
            preview = (msg.content or "")[:160].replace("\n", " ")
            return f"llm → final answer preview: {preview!r}"

    if node == "tools":
        msgs = payload.get("messages") or []
        bits: list[str] = []
        for m in msgs[-4:]:
            c = getattr(m, "content", "")
            bits.append(f"{m.__class__.__name__}:{str(c)[:120]!r}")
        return "tools → " + " | ".join(bits)

    return str(payload)[:240]


def log_state_transition(run_label: str, thread_id: str, step_index: int, update: dict) -> None:
    """Log each LangGraph update chunk as one line per node."""
    for node_name, delta in update.items():
        summary = summarize_update(node_name, delta)
        LOG.info(
            "[%s] step=%s thread=%s node=%s | %s",
            run_label,
            step_index,
            thread_id,
            node_name,
            summary,
        )


def run_tracked(run_label: str, thread_id: str, user_text: str, *, recursion_limit: int = 8) -> dict:
    """Stream the graph, log transitions, return node order + final checkpoint view."""
    cfg: dict = {"configurable": {"thread_id": thread_id}, "recursion_limit": recursion_limit}
    initial = {"messages": [HumanMessage(content=user_text)]}
    node_sequence: list[str] = []
    for step, update in enumerate(app.stream(initial, config=cfg, stream_mode="updates")):
        for node_name in update:
            node_sequence.append(node_name)
        log_state_transition(run_label, thread_id, step, update)
    final = app.get_state(cfg)
    return {"config": cfg, "node_sequence": tuple(node_sequence), "final_state": final}

## Two runs: same task, tiny wording change

We deliberately ask the model to **use the tool** so both runs are likely to visit `llm → tools → llm`. Small input changes can still flip behavior when temperature is non-zero or when the instruction is ambiguous—here **`temperature=0`** makes paths more repeatable, which is useful for a baseline.

In [9]:
PROMPT_A = (
    "Count the words in this sentence exactly using the tool: "
    "'LangGraph checkpointing helps debug agent workflows.'"
)
PROMPT_B = PROMPT_A.replace("exactly", "precisely")

r_a = run_tracked("run_a", "stability-demo-a", PROMPT_A)
r_b = run_tracked("run_b", "stability-demo-b", PROMPT_B)

2026-03-27 10:40:52,877 | INFO | [run_a] step=0 thread=stability-demo-a node=llm | llm → tool_calls=['word_stats']
INFO:workflow.debug:[run_a] step=0 thread=stability-demo-a node=llm | llm → tool_calls=['word_stats']
2026-03-27 10:40:52,884 | INFO | [run_a] step=1 thread=stability-demo-a node=tools | tools → ToolMessage:'{"chars": 52, "words": 6}'
INFO:workflow.debug:[run_a] step=1 thread=stability-demo-a node=tools | tools → ToolMessage:'{"chars": 52, "words": 6}'
2026-03-27 10:40:53,218 | INFO | [run_a] step=2 thread=stability-demo-a node=llm | llm → final answer preview: 'The sentence contains **6 words**.'
INFO:workflow.debug:[run_a] step=2 thread=stability-demo-a node=llm | llm → final answer preview: 'The sentence contains **6 words**.'
2026-03-27 10:40:53,675 | INFO | [run_b] step=0 thread=stability-demo-b node=llm | llm → tool_calls=['word_stats']
INFO:workflow.debug:[run_b] step=0 thread=stability-demo-b node=llm | llm → tool_calls=['word_stats']
2026-03-27 10:40:53,684 | INFO

## Compare transitions (stability)

Equality of **`node_sequence`** is a **structural** notion of stability (same nodes fired in the same order). The **final natural-language answer** can still differ slightly even when the path matches.

In [10]:
print("Prompt A:", PROMPT_A)
print("Prompt B:", PROMPT_B)
print()
print("Node sequence A:", r_a["node_sequence"])
print("Node sequence B:", r_b["node_sequence"])
print("Same path?", r_a["node_sequence"] == r_b["node_sequence"])

Prompt A: Count the words in this sentence exactly using the tool: 'LangGraph checkpointing helps debug agent workflows.'
Prompt B: Count the words in this sentence precisely using the tool: 'LangGraph checkpointing helps debug agent workflows.'

Node sequence A: ('llm', 'tools', 'llm')
Node sequence B: ('llm', 'tools', 'llm')
Same path? True


## Checkpoint history per `thread_id`

With a checkpointer compiled into the graph, LangGraph records **checkpoints** for each thread. `get_state_history` walks backward from the latest snapshot; below we **reverse** to print oldest → newest.

Use this to correlate **what the user said** with **how many LLM/tool turns** were persisted—handy when a log line and a support ticket do not obviously match.

In [11]:
def print_checkpoint_timeline(label: str, cfg: dict) -> None:
    hist = list(app.get_state_history(cfg))
    print(f"\n=== {label}: {len(hist)} checkpoint(s) ===")
    for i, snap in enumerate(reversed(hist)):
        vals = snap.values or {}
        msgs = vals.get("messages") or []
        last = msgs[-1] if msgs else None
        tail = ""
        if last is not None:
            tail = (getattr(last, "content", "") or "")[:100].replace("\n", " ")
        meta = getattr(snap, "metadata", None) or {}
        src = meta.get("source") if isinstance(meta, dict) else None
        print(f"  [{i:02d}] next={snap.next!r} messages={len(msgs)} source={src!r}")
        if tail:
            print(f"       last_msg_preview: {tail!r}")


print_checkpoint_timeline("thread A", r_a["config"])
print_checkpoint_timeline("thread B", r_b["config"])


=== thread A: 5 checkpoint(s) ===
  [00] next=('__start__',) messages=0 source='input'
  [01] next=('llm',) messages=1 source='loop'
       last_msg_preview: "Count the words in this sentence exactly using the tool: 'LangGraph checkpointing helps debug agent "
  [02] next=('tools',) messages=2 source='loop'
  [03] next=('llm',) messages=3 source='loop'
       last_msg_preview: '{"chars": 52, "words": 6}'
  [04] next=() messages=4 source='loop'
       last_msg_preview: 'The sentence contains **6 words**.'

=== thread B: 5 checkpoint(s) ===
  [00] next=('__start__',) messages=0 source='input'
  [01] next=('llm',) messages=1 source='loop'
       last_msg_preview: "Count the words in this sentence precisely using the tool: 'LangGraph checkpointing helps debug agen"
  [02] next=('tools',) messages=2 source='loop'
  [03] next=('llm',) messages=3 source='loop'
       last_msg_preview: '{"chars": 52, "words": 6}'
  [04] next=() messages=4 source='loop'
       last_msg_preview: 'The sentence c

## Persist checkpoints (SQLite)

For demos and small apps, `SqliteSaver` / `AsyncSqliteSaver` give you **durable** history without Postgres. The pattern is: create a saver, optionally call `setup()`, pass it to `compile(checkpointer=...)`, and keep using the same **`thread_id`** APIs.

```python
# from langgraph.checkpoint.sqlite import SqliteSaver
# import sqlite3
# conn = sqlite3.connect("checkpoints.db", check_same_thread=False)
# checkpointer = SqliteSaver(conn)
# app = workflow.compile(checkpointer=checkpointer)
```

**Takeaway:** use **`logging`** for live debugging in logs, and **checkpoint history** for post-hoc reconstruction of messages and graph position per conversation thread.